# Détroit d'Hormuz — AIS temps réel (AISStream.io)

**Bounding box Hormuz :** LAT [25.5, 27.0], LON [56.0, 59.0]

Ce notebook :
1. Se connecte au WebSocket AISStream.io
2. Filtre les messages dans le détroit d'Hormuz
3. Accumule les positions en DataFrame Polars
4. Génère une carte Folium interactive

**Prérequis :** mettre votre clé API dans `Nowcasting/.env` :
```
AISSTREAM_API_KEY=votre_clé_ici
```
Clé gratuite sur : https://aisstream.io

In [11]:
import os
from pathlib import Path

env_path = Path("..") / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line.strip() and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

API_KEY = os.environ.get("AISSTREAM_API_KEY", "")
if not API_KEY:
    raise ValueError("AISSTREAM_API_KEY manquante — ajoutez-la dans Nowcasting/.env")
print(f"Clé chargée : {API_KEY[:6]}...")

Clé chargée : 0cfcb9...


In [12]:
# Bounding box Détroit d'Hormuz
LAT_MIN, LAT_MAX = 25.5, 27.0
LON_MIN, LON_MAX = 56.0, 59.0

COLLECT_SECONDS = 120

print(f"Zone : LAT [{LAT_MIN}, {LAT_MAX}] | LON [{LON_MIN}, {LON_MAX}]")
print(f"Durée : {COLLECT_SECONDS}s")

Zone : LAT [25.5, 27.0] | LON [56.0, 59.0]
Durée : 120s


In [13]:
# ── DIAGNOSTIC — teste la connexion sur la Manche (couverture garantie, 15s)
import asyncio, json as _j, websockets

async def diagnose():
    # Pas de FilterMessageTypes — seuls APIKey et BoundingBoxes sont obligatoires
    sub = {"APIKey": API_KEY, "BoundingBoxes": [[[49.0, -5.0], [52.0, 5.0]]]}
    async with websockets.connect("wss://stream.aisstream.io/v0/stream") as ws:
        await ws.send(_j.dumps(sub))
        print("Subscription envoyée — attente...")
        count = 0
        async for raw in ws:
            msg = _j.loads(raw)
            pos = msg.get("Message", {}).get("PositionReport", {})
            meta = msg.get("Metadata", {})
            print(f"[{count+1}] type={msg.get('MessageType')} "
                  f"lat={pos.get('Latitude')} lon={pos.get('Longitude')} "
                  f"ship={meta.get('ShipName', '?')}")
            count += 1
            if count >= 5:
                break
    print(f"\n{'OK — connexion fonctionnelle' if count else 'ECHEC — 0 message reçu'}")

await diagnose()

Subscription envoyée — attente...
[1] type=PositionReport lat=51.81996 lon=4.714345 ship=?
[2] type=StandardClassBPositionReport lat=None lon=None ship=?
[3] type=PositionReport lat=50.837808333333335 lon=-1.3294666666666666 ship=?
[4] type=PositionReport lat=51.822161666666666 lon=4.0407633333333335 ship=?
[5] type=PositionReport lat=51.826555 lon=4.03535 ship=?

OK — connexion fonctionnelle


In [14]:
import asyncio
import json
import time
from datetime import datetime, timezone

import websockets
from IPython.display import clear_output

AISSTREAM_URL = "wss://stream.aisstream.io/v0/stream"

records = []
_raw_count = 0

async def collect_hormuz(duration_seconds: int) -> list[dict]:
    global _raw_count
    _raw_count = 0

    subscription = {
        "APIKey": API_KEY,
        "BoundingBoxes": [[[LAT_MIN, LON_MIN], [LAT_MAX, LON_MAX]]],
        # NB : pas de FilterMessageTypes — un type invalide bloque silencieusement
        # On filtre par type dans le code ci-dessous
    }

    collected = []
    t_start = time.time()

    def _show(elapsed, msg=None):
        clear_output(wait=True)
        print(f"Collecte... [{elapsed:3d}s / {duration_seconds}s]")
        print(f"  Messages bruts reçus : {_raw_count}")
        print(f"  Positions dans bbox  : {len(collected)}")
        if msg:
            print(f"  {msg}")

    print("Connexion à AISStream.io...")
    async with websockets.connect(AISSTREAM_URL) as ws:
        await ws.send(json.dumps(subscription))
        _show(0, "Abonné — en attente de messages...")

        while time.time() - t_start < duration_seconds:
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                msg = json.loads(raw)
                _raw_count += 1

                # Filtrer uniquement les PositionReport (Class A)
                if msg.get("MessageType") != "PositionReport":
                    continue

                # Doc AISStream : clé "Metadata" (d minuscule)
                meta = msg.get("Metadata", {})
                pos  = msg.get("Message", {}).get("PositionReport", {})

                # Latitude/Longitude dans Metadata (uppercase) ou dans PositionReport
                lat = meta.get("Latitude") or pos.get("Latitude")
                lon = meta.get("Longitude") or pos.get("Longitude")

                if lat is None or lon is None:
                    continue
                lat, lon = float(lat), float(lon)

                if not (LAT_MIN <= lat <= LAT_MAX and LON_MIN <= lon <= LON_MAX):
                    continue

                record = {
                    "MMSI":       pos.get("UserID") or meta.get("MMSI"),
                    "VesselName": meta.get("ShipName", "").strip(),
                    "LAT":        lat,
                    "LON":        lon,
                    "SOG":        float(pos.get("Sog") or 0),
                    "COG":        float(pos.get("Cog") or 0),
                    "Heading":    pos.get("TrueHeading"),
                    "NavStatus":  pos.get("NavigationalStatus"),
                    "timestamp":  meta.get("TimeUtc", datetime.now(timezone.utc).isoformat()),
                }
                collected.append(record)

                elapsed = int(time.time() - t_start)
                last = collected[-1]
                _show(elapsed, f"Dernier : {last['VesselName'] or last['MMSI']}  SOG={last['SOG']:.1f} kt")

            except asyncio.TimeoutError:
                elapsed = int(time.time() - t_start)
                _show(elapsed, "En attente de messages...")
            except websockets.exceptions.ConnectionClosed:
                print("\nConnexion fermée par le serveur")
                break

    clear_output(wait=True)
    print(f"✓ Collecte terminée : {len(collected)} positions | {_raw_count} messages bruts")
    return collected


records = await collect_hormuz(COLLECT_SECONDS)

✓ Collecte terminée : 0 positions | 0 messages bruts


In [15]:
import polars as pl

if not records:
    print("Aucun message reçu — relancez la collecte (cell 4)")
else:
    df = pl.DataFrame(records)
    print(f"Shape : {df.shape}")
    print(f"Navires distincts : {df['MMSI'].n_unique()}")
    sog = df.filter(pl.col("SOG").is_not_null())["SOG"]
    print(f"SOG — min={sog.min():.1f}  max={sog.max():.1f}  mean={sog.mean():.1f} kt")
    print()
    print(df.select(["MMSI", "VesselName", "LAT", "LON", "SOG", "NavStatus"]).head(10))

Aucun message reçu — relancez la collecte (cell 4)


In [16]:
# Dédupliquer : garder la dernière position par MMSI
if not records:
    print("Aucune donnée — relancez la cellule de collecte")
else:
    df_last = (
        df
        .sort("timestamp")
        .group_by("MMSI")
        .agg([
            pl.col("VesselName").last(),
            pl.col("LAT").last(),
            pl.col("LON").last(),
            pl.col("SOG").last(),
            pl.col("COG").last(),
            pl.col("NavStatus").last(),
            pl.col("timestamp").last(),
            pl.len().alias("n_messages"),
        ])
    )
    print(f"Navires uniques (dernière position) : {len(df_last)}")
    display(df_last.head())

Aucune donnée — relancez la cellule de collecte


In [17]:
if not records:
    print("Aucune donnée — relancez la cellule de collecte")
else:
    import folium

    NAV_STATUS = {
        0: "En route (moteur)", 1: "Au mouillage", 2: "Non commandé",
        3: "Manœuvrabilité réduite", 5: "Amarré", 8: "En route (voile)", 15: "Non défini",
    }

    def vessel_color(sog, nav_status):
        if nav_status in (1, 5): return "blue"
        if sog is None:          return "gray"
        if sog < 1:              return "orange"
        if sog < 10:             return "green"
        return "red"

    center = [(LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2]
    m = folium.Map(location=center, zoom_start=9, tiles="CartoDB positron")

    folium.Rectangle(
        bounds=[[LAT_MIN, LON_MIN], [LAT_MAX, LON_MAX]],
        color="#1565C0", fill=False, weight=2, dash_array="6 4",
        tooltip="Détroit d'Hormuz",
    ).add_to(m)

    for row in df_last.iter_rows(named=True):
        sog   = row["SOG"]
        ns    = row["NavStatus"]
        color = vessel_color(sog, ns)
        name  = row["VesselName"] or str(row["MMSI"])
        status = NAV_STATUS.get(ns, f"Status {ns}")
        folium.CircleMarker(
            location=[row["LAT"], row["LON"]],
            radius=5, color=color, fill=True, fill_opacity=0.75,
            popup=folium.Popup(
                f"<b>{name}</b><br>MMSI : {row['MMSI']}<br>"
                f"SOG : {sog:.1f} kt<br>COG : {row['COG']:.0f}°<br>"
                f"Status : {status}<br>Messages : {row['n_messages']}",
                max_width=200,
            ),
            tooltip=f"{name} | {sog:.1f} kt",
        ).add_to(m)

    legend = (
        '<div style="position:fixed;bottom:30px;left:30px;z-index:999;'
        'background:white;padding:10px 14px;border-radius:8px;'
        'border:1px solid #ccc;font-family:sans-serif;font-size:12px">'
        "<b>Détroit d'Hormuz — AIS temps réel</b><br><br>"
        '<span style="color:blue">●</span> Ancré / Amarré<br>'
        '<span style="color:orange">●</span> Quasi-stationnaire (&lt;1 kt)<br>'
        '<span style="color:green">●</span> Lent (1–10 kt)<br>'
        '<span style="color:red">●</span> Rapide (&gt;10 kt)<br>'
        '<span style="color:gray">●</span> Inconnu</div>'
    )
    m.get_root().html.add_child(folium.Element(legend))

    out_path = "../outputs/figures/hormuz_realtime.html"
    m.save(out_path)
    print(f"Carte sauvegardée : {out_path}")
    m

Aucune donnée — relancez la cellule de collecte


In [18]:
if not records:
    print("Aucune donnée — relancez la cellule de collecte")
else:
    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d_%H%M")
    out_parquet = f"../data/parquet/hormuz_realtime_{ts}.parquet"
    df.write_parquet(out_parquet)
    print(f"Données brutes sauvegardées : {out_parquet}")
    print(f"  {len(df)} messages | {df['MMSI'].n_unique()} navires")

Aucune donnée — relancez la cellule de collecte
